# Recording parameter access

QCoDeS can record every `get` and `set` of every parameter that a piece of code makes. Each access is recorded as a `ParameterAccess`: the names of the parameter and its instrument, the value and raw value, when the access started, how long it took, and the error it raised, if any.

This lets you see what a piece of code did to your instruments without changing the code or the drivers it uses. You can record what a measurement set and read, find out which parameters are queried most often, or keep a log of everything that was set.

Recording only stores what happened:

- Accesses are appended to a plain `list`, or logged to a fixed QCoDeS logger. No code of yours runs while a parameter is accessed, so recording cannot be used to react to an access, for example by setting another parameter.
- A record holds data only: the names of the parameter and its instrument rather than the objects, and a description of an error rather than the exception.
- Only accesses made in the context recording was started in are recorded: the current thread, or the current asyncio task.

In [1]:
import logging
import sys
import tempfile
import threading
from collections import Counter
from pathlib import Path

from qcodes.dataset import (
    do1d,
    initialise_or_create_database_at,
    load_or_create_experiment,
)
from qcodes.instrument_drivers.mock_instruments import (
    DummyInstrument,
    DummyInstrumentWithMeasurement,
)
from qcodes.parameters import (
    Parameter,
    ParameterAccess,
    log_parameter_accesses,
    record_parameter_accesses,
)

We use two mock instruments: a DAC with two channels, and a DMM whose readings depend on the DAC's voltages.

In [2]:
dac = DummyInstrument("dac", gates=["ch1", "ch2"])
dmm = DummyInstrumentWithMeasurement("dmm", setter_instr=dac)

## Recording the accesses of a block of code

`record_parameter_accesses` records the accesses made inside a `with` block in a list, in the order they complete.

In [3]:
with record_parameter_accesses() as accesses:
    dac.ch1(0.5)
    dmm.v1()

for access in accesses:
    print(f"{access.kind} {access.parameter_name}: {access.value}")

set dac_ch1: 0.5
get dmm_v1: 4.3682652935244874


Accesses outside the block are not recorded:

In [4]:
dac.ch1(0.7)
len(accesses)

2

You can also pass in the list to record in. It has to be a plain `list`: recording only ever appends to it.

In [5]:
everything: list[ParameterAccess] = []
with record_parameter_accesses(everything):
    dac.ch2(0.1)
with record_parameter_accesses(everything):
    dac.ch2(0.2)

[access.value for access in everything]

[0.1, 0.2]

## What a record contains

Each `ParameterAccess` describes one access:

- `parameter_name`: the full name of the parameter.
- `instrument_name`: the full name of the parameter's root instrument, or `None`.
- `kind`: `"get"` or `"set"`.
- `value`: the value that was read or written.
- `raw_value`: the value passed to or returned by the instrument, before scaling, offsets or value mappings are applied.
- `unit`: the parameter's unit.
- `timestamp`: when the access started, in seconds since the epoch.
- `duration`: how long the access took, in seconds. Delays configured on the parameter, such as `post_delay`, are not included.
- `error`: the type and message of the exception the access raised, or `None`.

The difference between `value` and `raw_value` shows what actually reached the instrument. For example, behind a 1:10 voltage divider, setting 2 V on the sample means setting 20 V on the source:

In [6]:
divided = Parameter("divided", unit="V", set_cmd=None, get_cmd=None, scale=10)

with record_parameter_accesses() as set_accesses:
    divided(2.0)

(access,) = set_accesses
print(f"value={access.value}, raw_value={access.raw_value}, error={access.error}")

value=2.0, raw_value=20.0, error=None


## Ramped sets are recorded step by step

A parameter with a `step` is set in several steps, each of which is a separate call to the instrument. Every step is recorded:

In [7]:
dac.ch2.step = 0.1
dac.ch2.inter_delay = 0

with record_parameter_accesses() as ramp:
    dac.ch2(0.5)

[round(access.value, 2) for access in ramp]

[0.3, 0.4, 0.5]

## Failed accesses

An access that raises is recorded too, with a description of the exception in `error`, and the exception then propagates to the caller as usual. A value that fails validation is rejected before the instrument is contacted, so it is not an access and is not recorded.

In [8]:
def read_unresponsive_sensor() -> float:
    raise TimeoutError("the sensor did not answer")


sensor = Parameter("sensor", get_cmd=read_unresponsive_sensor)

with record_parameter_accesses() as failures:
    try:
        sensor()
    except TimeoutError as error:
        print(f"The caller still sees the error: {error.args[0]}")

(failure,) = failures
print(f"Recorded: {failure.kind} {failure.parameter_name}: {failure.error}")

The caller still sees the error: the sensor did not answer
Recorded: get sensor: TimeoutError: the sensor did not answer


## What did a measurement do to the instruments?

Recording sees all accesses made in its context, including those QCoDeS makes itself, so it can show what a measurement did. Here we run a 1D sweep and count the accesses per parameter.

In [9]:
initialise_or_create_database_at(Path(tempfile.mkdtemp()) / "recording.db")
load_or_create_experiment("recording_parameter_access", sample_name="no sample")

with record_parameter_accesses() as sweep:
    do1d(dac.ch1, 0, 1, 11, 0.0, dmm.v1, do_plot=False)

Counter((access.parameter_name, access.kind) for access in sweep)

Starting experimental run with id: 1. Using 'qcodes.dataset.do1d'


Counter({('dac_ch1', 'set'): 11, ('dmm_v1', 'get'): 11})

The sweep set the DAC 11 times and read the DMM 11 times. The accesses are in the order they happened, so we can also follow the sweep point by point:

In [10]:
for access in sweep[:6]:
    print(f"{access.kind} {access.parameter_name}: {access.value:.3f}")

set dac_ch1: 0.000
get dmm_v1: 4.946
set dac_ch1: 0.100
get dmm_v1: 4.862
set dac_ch1: 0.200
get dmm_v1: 4.809


To follow one instrument, filter on the instrument name:

In [11]:
dac_sets = [
    access.value
    for access in sweep
    if access.instrument_name == "dac" and access.kind == "set"
]
[round(float(value), 2) for value in dac_sets]

[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

## Recording is local to a thread

Only accesses made by the thread (or asyncio task) that started recording are recorded. Work in other threads, even while the block runs, is not:

In [12]:
with record_parameter_accesses() as here:
    worker = threading.Thread(target=dac.ch1, args=(0.9,))
    worker.start()
    worker.join()
    dac.ch1(0.3)

[access.value for access in here]

[0.3]

Recordings can be nested. Each one receives every access made while it is active:

In [13]:
with record_parameter_accesses() as outer:
    dac.ch1(0.1)
    with record_parameter_accesses() as inner:
        dac.ch1(0.2)

[access.value for access in outer], [access.value for access in inner]

([0.1, 0.2], [0.2])

## Logging accesses

`log_parameter_accesses` logs the accesses made inside a `with` block to the `qcodes.parameters.parameter_access` logger, at `DEBUG` level unless you choose another. Each log record carries the `ParameterAccess` as its `parameter_access` attribute, so a handler can store it in structured form.

In [14]:
access_log = logging.getLogger("qcodes.parameters.parameter_access")
access_log.setLevel(logging.DEBUG)
access_log.propagate = False
access_log.addHandler(logging.StreamHandler(sys.stdout))

with log_parameter_accesses():
    dac.ch1(0.25)
    reading = dmm.v2()

set dac_ch1 = 0.25


get dmm_v2 = 0.4612975233041446


## Good to know

- Recording adds a little work to every access made while it is active, and none otherwise.
- Values are recorded as the parameter returned or received them, not copied.
- An access to a `DelegateParameter` is recorded twice: once for the delegate and once for its source.
- Only parameters are recorded. Commands that a driver sends directly, for example with `Instrument.write` or `Instrument.ask` in a driver method, are not. Use the instrument's logger (`visa_log` for VISA instruments) to see the communication on the wire.

In [15]:
dac.close()
dmm.close()